# 🐝 Lab 5.4 — Peeking Inside: Sparse Autoencoders & Steering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab16_sae_interpretability.ipynb)

⏱ **Time:** ~2 hours  |  💻 **Runs on:** CPU (GPU just makes the SAE train faster)

### 🎯 Learning objectives
- Explain the **residual stream** view of a transformer and use the **logit lens** to watch a prediction form layer by layer.
- Collect residual-stream activations from **GPT-2 small** with forward hooks.
- Train a **sparse autoencoder** (TopK, with a ReLU+L1 baseline) and track **L0**, **fraction of variance explained** and **dead latents**.
- Interpret SAE features through their **top-activating contexts** and the tokens their decoder directions promote, and test them by **splicing the SAE into the model**.
- **Steer** generation with a contrastive mean-difference vector and with an SAE feature direction.

### 🧰 Prerequisites
PyTorch modules and hooks (we'll explain), autoencoders at the "encode → bottleneck → decode" level, and a basic picture of a GPT-style transformer (Lab 3.1 "A Modern GPT From Scratch" is ideal).

### 🌍 Why this matters in 2026
As models take on more responsibility, "it scores well on the benchmark" stops being enough — we want to know *what they're computing*. **Dictionary learning with sparse autoencoders** turned out to be one of the most productive tools for this: Anthropic's *Towards Monosemanticity* (2023) and *Scaling Monosemanticity* (2024) found millions of interpretable features inside production models, and OpenAI and Google DeepMind released large open SAE suites. Those features can be used to **audit**, **monitor** and **steer** models. Buzz the Backprop Bee 🐝 will build the whole pipeline on GPT-2 small in an afternoon.

## 🗺️ Roadmap
| Part | What you'll do | ⏱ min |
|---|---|---|
| 0 | Setup: load GPT-2 small | 5 |
| 1 | Logit lens: watch a prediction form layer by layer (heatmap and animation) | 20 |
| 2 | Harvest residual-stream activations with forward hooks | 10 |
| 3 | Train TopK and ReLU+L1 sparse autoencoders; track L0, FVE and dead latents | 30 |
| 4 | Read features: density, top-activating contexts, promoted tokens | 20 |
| 5 | Splice the SAE into the model and measure the loss recovered | 10 |
| 6 | Steering with a mean-difference vector and with an SAE feature | 25 |

## 0 · Setup
`transformers` and `datasets` are preinstalled on Colab. GPT-2 small is a ~550 MB download (FAST mode uses the ~350 MB `distilgpt2`).

In [ ]:
import os, sys, math, random, time, json, re
import numpy as np
import torch
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (normal users: leave unset)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed: int = 42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA",
                     "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})
print(f"🐝 torch {torch.__version__} | device={DEVICE} | FAST={FAST}")

import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", min(8, os.cpu_count() or 1))))
import warnings
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
try:
    from huggingface_hub.utils import logging as hub_logging
    hub_logging.set_verbosity_error()
    import datasets; datasets.disable_progress_bars()
except Exception:
    pass

MODEL_NAME = "distilgpt2" if FAST else "gpt2"
N_TOKENS   = 10_000 if FAST else 60_000     # activations to collect
SEQ_LEN    = 128
EXPANSION  = 4 if FAST else 8               # SAE width = EXPANSION * d_model
TOPK_K     = 32
SAE_STEPS  = 300 if FAST else 2000
RELU_STEPS = 200 if FAST else 1000
BATCH      = 512

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).float().to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)
N_LAYERS, D_MODEL = model.config.n_layer, model.config.n_embd
LAYER = N_LAYERS // 2                        # residual stream right after this block
print(f"{MODEL_NAME}: {N_LAYERS} blocks, d_model={D_MODEL}; we study the residual stream after block {LAYER}")

## Part 1 · Warm-up: the residual stream and the logit lens 🔭

A GPT-2 block *adds* to a running vector — the **residual stream**:

$$h_{\ell+1} = h_\ell + \text{Attn}_\ell(h_\ell) + \text{MLP}_\ell(h_\ell + \text{Attn}_\ell(h_\ell)), \qquad \text{logits} = W_U\,\text{LN}_f(h_L).$$

Every layer reads from and writes to the same $d_\text{model}$-dimensional space, and only the end of the stream gets decoded. The **logit lens** (nostalgebraist, 2020) asks: what if we decoded *early*? Apply the final layer norm and unembedding to $h_\ell$ for every $\ell$ and see which token the model "would" predict at that depth.

### 🧪 Your Turn #1 — implement the logit lens
`hidden_states` is the tuple returned with `output_hidden_states=True`: entry 0 is the embedding output, entry $\ell$ the stream after block $\ell$. **Careful:** in Hugging Face GPT-2 the *last* entry has already been passed through `ln_f`, so don't apply it twice.

In [ ]:
def logit_lens(hidden_states):
    """Return log-probs of shape (n_layers+1, T, vocab) for a single sequence (batch index 0)."""
    # TODO: for each h in hidden_states[:-1]: logits = model.lm_head(model.transformer.ln_f(h[0]))
    # TODO: for the last entry: logits = model.lm_head(hidden_states[-1][0])   (ln_f already applied)
    # TODO: stack and log_softmax over the vocab
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
@torch.no_grad()
def logit_lens(hidden_states):
    outs = [model.lm_head(model.transformer.ln_f(h[0])) for h in hidden_states[:-1]]
    outs.append(model.lm_head(hidden_states[-1][0]))
    return torch.log_softmax(torch.stack(outs), dim=-1)

In [ ]:
# ✔️ Checkpoint
prompt = "The Eiffel Tower is located in the city of"
enc = tok(prompt, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    out = model(**enc, output_hidden_states=True)
lens = logit_lens(out.hidden_states)
assert lens.shape == (N_LAYERS + 1, enc.input_ids.shape[1], model.config.vocab_size)
assert torch.allclose(lens[-1], torch.log_softmax(out.logits[0], -1), atol=1e-4), "last layer of the lens == the model's output"
print("✔️ logit lens reproduces the final logits")

target = tok(" Paris").input_ids[0]
p_paris = lens[:, -1, target].exp().cpu().numpy()
rank = (lens[:, -1] > lens[:, -1, target:target + 1]).sum(-1).cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(p_paris, "o-"); ax[0].set_title("P(' Paris') at the last position, by layer"); ax[0].set_xlabel("layer (0 = embeddings)")
ax[1].plot(rank + 1, "o-", color=HIVE[2]); ax[1].set_yscale("log"); ax[1].invert_yaxis()
ax[1].set_title("rank of ' Paris' (1 = top prediction)"); ax[1].set_xlabel("layer")
plt.tight_layout(); plt.show()

In [ ]:
# Top-1 decoded token at every (layer, position)
toks = [tok.decode([t]) for t in enc.input_ids[0]]
top = lens.argmax(-1).cpu().numpy(); topp = lens.max(-1).values.exp().cpu().numpy()
fig, ax = plt.subplots(figsize=(1.15 * len(toks) + 2, 0.42 * (N_LAYERS + 1) + 1.5))
ax.imshow(topp, aspect="auto", cmap="YlOrBr", vmin=0, vmax=1)
for l in range(top.shape[0]):
    for t in range(top.shape[1]):
        ax.text(t, l, repr(tok.decode([top[l, t]]))[1:-1][:8], ha="center", va="center", fontsize=7,
                color="white" if topp[l, t] > 0.55 else "black")
ax.set_xticks(range(len(toks)), [repr(t)[1:-1] for t in toks], rotation=30, fontsize=8)
ax.set_ylabel("layer"); ax.set_title("Logit lens: top-1 next-token guess (colour = probability)")
plt.tight_layout(); plt.show()

### 🎬 The prediction, layer by layer
The same lens as an animation: at the **last position** of the prompt, the top-8 candidate next tokens and their probabilities as we decode deeper and deeper into the network. Watch candidates rise and fall, and see when the final answer takes over.

In [ ]:
from matplotlib import animation
from IPython.display import HTML, display

probs_last = lens[:, -1].exp().cpu()                       # (layers+1, vocab)
cands = torch.unique(probs_last.topk(3, dim=-1).indices.flatten())[:14]   # tokens that are ever top-3
labels = [repr(tok.decode([t]))[1:-1] for t in cands.tolist()]
fig, ax = plt.subplots(figsize=(9, 4.2))
def update(l):
    ax.clear()
    p = probs_last[l, cands].numpy()
    order = np.argsort(p)[-8:]
    ax.barh(range(len(order)), p[order], color=[HIVE[3] if labels[i].strip() == "Paris" else HIVE[0] for i in order])
    ax.set_yticks(range(len(order)), [labels[i] for i in order])
    ax.set_xlim(0, max(0.05, float(probs_last[:, cands].max())) * 1.05)
    ax.set_title(f"'{prompt} ___'   decoded after layer {l}/{N_LAYERS}")
    ax.set_xlabel("probability")
anim = animation.FuncAnimation(fig, update, frames=N_LAYERS + 1, interval=600)
display(HTML(anim.to_jshtml()))
plt.close(fig)

Early layers mostly echo the current token or produce generic continuations; the right answer typically surfaces in the second half of the network and sharpens toward the end. With GPT-2 small, watch the last column: the guess wanders through other places (a country, other cities) before settling on Paris only in the final layers. (The lens is a rough tool — later variants such as the *tuned lens*, Belrose et al. 2023, learn a small affine probe per layer.) The middle of the stream is where a lot of abstract information lives, which is why we'll train our SAE there.

> **🍯 Key takeaway:** The residual stream is a shared workspace that every block reads from and adds to. The logit lens decodes it early, and the answer typically crystallizes in the later layers.

## Part 2 · Harvesting activations with forward hooks 🪝

A **forward hook** is a function PyTorch calls with `(module, inputs, output)` every time a module runs. Returning a value *replaces* the output — we'll use that for splicing and steering later. Here we just record the residual stream after block `LAYER`.

Corpus: WikiText-2 (via `datasets`; falls back to a small inline text if offline). We drop position 0 of each sequence — the first token acts as an "attention sink" with a huge activation norm that would dominate the SAE's reconstruction loss.

In [ ]:
FALLBACK_TEXT = """Honey bees live in colonies with a single queen, thousands of female workers and a few hundred male drones.
Workers forage for nectar and pollen, build wax comb, nurse larvae and defend the hive. A forager that finds a rich
patch of flowers returns and performs a waggle dance that encodes direction and distance. The transformer is a neural
network architecture built from attention layers and multilayer perceptrons connected by a residual stream. Language
models are trained to predict the next token in a sequence of text. The city of Paris is the capital of France and
the Eiffel Tower was completed in 1889. The storm made landfall on the coast with winds of over one hundred miles per
hour, destroying homes and flooding roads. The band released its second album in 1994 and toured across Europe.
""" * 40

def load_corpus_text():
    try:
        from datasets import load_dataset
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
        text = "\n".join(x for x in ds["text"] if len(x.strip()) > 50)
        print(f"WikiText-2 loaded: {len(text)/1e6:.1f}M characters")
        return text
    except Exception as e:
        print("⚠️ could not load WikiText-2, using the inline fallback text:", type(e).__name__)
        return FALLBACK_TEXT

text = load_corpus_text()
ids = tok(text[: N_TOKENS * 8])["input_ids"][: N_TOKENS + N_TOKENS // SEQ_LEN + SEQ_LEN]
n_seq = len(ids) // SEQ_LEN
seqs = torch.tensor(ids[: n_seq * SEQ_LEN]).view(n_seq, SEQ_LEN)
print("sequences:", tuple(seqs.shape))

def resid_hook_output(output):
    return output[0] if isinstance(output, tuple) else output      # blocks return a tuple in older versions

@torch.no_grad()
def collect_resid(seqs, layer=LAYER, batch=16):
    store = []
    h = model.transformer.h[layer].register_forward_hook(lambda m, i, o: store.append(resid_hook_output(o).detach().cpu()))
    try:
        for i in range(0, len(seqs), batch):
            model(seqs[i:i + batch].to(DEVICE))
    finally:
        h.remove()                                  # always remove hooks!
    return torch.cat(store)                         # (n_seq, SEQ_LEN, d_model)

t0 = time.time()
resid = collect_resid(seqs)
print(f"collected {tuple(resid.shape)} in {time.time()-t0:.0f}s")

norms = resid.norm(dim=-1)
plt.figure(figsize=(7, 3)); plt.plot(norms.mean(0)[:40].numpy(), "o-")
plt.title(f"mean residual norm by position (layer {LAYER})"); plt.xlabel("position"); plt.yscale("log"); plt.show()

acts = resid[:, 1:].reshape(-1, D_MODEL)              # drop position 0
act_tokens = seqs[:, 1:].reshape(-1)
act_pos = torch.arange(1, SEQ_LEN).repeat(n_seq)
act_seq = torch.arange(n_seq).repeat_interleave(SEQ_LEN - 1)
ACT_MEAN = acts.mean(0)
ACT_SCALE = (acts - ACT_MEAN).norm(dim=-1).mean()    # so normalized activations have mean norm 1
X = ((acts - ACT_MEAN) / ACT_SCALE).to(DEVICE)
print(f"training activations: {tuple(X.shape)} | raw mean norm {acts.norm(dim=-1).mean():.1f} | scale {ACT_SCALE:.1f}")

> **🍯 Key takeaway:** Forward hooks let you read any intermediate activation without touching model code. Drop the attention-sink position 0, because its huge norm would dominate the SAE's loss.

## Part 3 · Sparse autoencoders 🧩

**The superposition hypothesis** (Elhage et al., 2022): models represent *more features than they have dimensions* by storing them as nearly-orthogonal directions, each active only rarely. Individual neurons are therefore **polysemantic**. A sparse autoencoder tries to recover the underlying features: a wide dictionary ($m \gg d$) where each activation is explained by **a few** active latents.

$$z = \sigma\big(W_\text{enc}(x - b_\text{dec}) + b_\text{enc}\big) \in \mathbb{R}^m, \qquad \hat x = W_\text{dec}^\top z + b_\text{dec}, \qquad \lVert W_{\text{dec},i}\rVert = 1.$$

Two ways to make $z$ sparse:

| | **ReLU + L1** (Bricken et al., 2023) | **TopK** (Gao et al., 2024) |
|---|---|---|
| activation $\sigma$ | ReLU | keep the $k$ largest pre-activations, ReLU them, zero the rest |
| loss | $\lVert x-\hat x\rVert^2 + \lambda \lVert z\rVert_1$ | $\lVert x-\hat x\rVert^2$ only |
| sparsity | tuned indirectly via $\lambda$ | **exactly** $L_0 = k$ |
| known issue | L1 *shrinks* activations | needs tricks against dead latents at scale |

Metrics we track:
- **L0** = average number of active latents per token (lower = sparser, more interpretable),
- **FVE** (fraction of variance explained) $= 1 - \frac{\sum\lVert x-\hat x\rVert^2}{\sum\lVert x-\bar x\rVert^2}$,
- **dead latents** = fraction of latents that never fired over a recent window of tokens.

### 🧪 Your Turn #2 — the TopK activation
Given pre-activations `pre` of shape `(B, m)`, return a tensor of the same shape that keeps the top-`k` entries of each row (after ReLU) and zeros everything else. *Hint:* `pre.topk(k, dim=-1)` and `torch.zeros_like(pre).scatter(-1, idx, vals)`.

In [ ]:
def topk_activation(pre, k):
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def topk_activation(pre, k):
    vals, idx = pre.topk(k, dim=-1)
    return torch.zeros_like(pre).scatter(-1, idx, F.relu(vals))

In [ ]:
# ✔️ Checkpoint
pre = torch.tensor([[0.5, -1.0, 2.0, 0.1], [-3.0, -2.0, -1.0, 4.0]])
z = topk_activation(pre, 2)
assert torch.equal(z, torch.tensor([[0.5, 0.0, 2.0, 0.0], [0.0, 0.0, 0.0, 4.0]])), z   # ReLU kills the negative "top" value
assert ((topk_activation(torch.randn(64, 100), 7) != 0).sum(-1) <= 7).all()
print("✔️ TopK activation OK")

In [ ]:
class SAE(nn.Module):
    def __init__(self, d, m, kind="topk", k=TOPK_K):
        super().__init__()
        self.kind, self.k = kind, k
        W_dec = F.normalize(torch.randn(m, d), dim=1)
        self.W_dec = nn.Parameter(W_dec)                          # (m, d) — row i is feature i's direction
        self.W_enc = nn.Parameter(W_dec.T.clone() * (1.0 if kind == "topk" else 0.1))   # tied init
        self.b_enc = nn.Parameter(torch.zeros(m))
        self.b_dec = nn.Parameter(torch.zeros(d))

    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        return topk_activation(pre, self.k) if self.kind == "topk" else F.relu(pre)

    def decode(self, z):
        return z @ self.W_dec + self.b_dec

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z

    @torch.no_grad()
    def normalize_decoder(self):
        self.W_dec.data = F.normalize(self.W_dec.data, dim=1)

### 🧪 Your Turn #3 — SAE metrics
Implement `sae_metrics(x, x_hat, z)` returning a dict with `fve` (float), `l0` (float: mean number of non-zero latents per row) and `fired` (bool tensor of shape `(m,)`: did each latent fire at least once in this batch?).

In [ ]:
def sae_metrics(x, x_hat, z):
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def sae_metrics(x, x_hat, z):
    fve = 1 - (x - x_hat).pow(2).sum() / (x - x.mean(0)).pow(2).sum()
    return {"fve": fve.item(), "l0": (z > 0).float().sum(-1).mean().item(), "fired": (z > 0).any(0)}

In [ ]:
# ✔️ Checkpoint
x = torch.randn(100, 8)
m_ = sae_metrics(x, x, torch.tensor([[1.0, 0, 0], [0, 2.0, 0]]).repeat(50, 1))
assert abs(m_["fve"] - 1.0) < 1e-6 and abs(m_["l0"] - 1.0) < 1e-6 and m_["fired"].tolist() == [True, True, False]
assert abs(sae_metrics(x, x.mean(0).expand_as(x), torch.zeros(100, 3))["fve"]) < 1e-6, "predicting the mean explains 0 variance"
print("✔️ metrics OK")

In [ ]:
def train_sae(kind, steps, l1=0.0, lr=1e-3, seed=0, window=200):
    seed_everything(seed)
    sae = SAE(D_MODEL, EXPANSION * D_MODEL, kind=kind).to(DEVICE)
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    hist = {"step": [], "fve": [], "l0": [], "dead": []}
    fired = torch.zeros(sae.W_dec.shape[0], dtype=torch.bool, device=DEVICE)
    t0 = time.time()
    for step in range(steps):
        x = X[torch.randint(0, len(X), (BATCH,), device=DEVICE)]
        x_hat, z = sae(x)
        mse = (x_hat - x).pow(2).sum(-1).mean()
        loss = mse + l1 * z.abs().sum(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        sae.normalize_decoder()                      # keep decoder rows unit-norm (so L1 can't cheat by shrinking z)
        with torch.no_grad():
            m_ = sae_metrics(x, x_hat, z); fired |= m_["fired"]
        if (step + 1) % window == 0 or step == steps - 1:
            hist["step"].append(step); hist["fve"].append(m_["fve"]); hist["l0"].append(m_["l0"])
            hist["dead"].append(1 - fired.float().mean().item()); fired.zero_()
    print(f"{kind:5s} SAE: {steps} steps in {time.time()-t0:.0f}s | FVE {hist['fve'][-1]:.3f} | L0 {hist['l0'][-1]:.1f} | dead {hist['dead'][-1]:.1%}")
    return sae, hist

### 🔮 Predict-then-run
The TopK SAE has $m = 8 \times 768 = 6144$ latents (4× in FAST mode) and uses exactly 32 per token. Roughly what fraction of the variance of GPT-2's middle-layer residual stream do you think 32 dictionary directions per token can explain? 30%? 60%? 90%?

In [ ]:
sae, hist = train_sae("topk", SAE_STEPS)
relu_sae, relu_hist = train_sae("relu", RELU_STEPS, l1=0.1)   # λ tuned so L0 lands near TopK's k

fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
for h, name, col in [(hist, f"TopK (k={TOPK_K})", HIVE[0]), (relu_hist, "ReLU + L1", HIVE[4])]:
    ax[0].plot(h["step"], h["fve"], "o-", color=col, label=name)
    ax[1].plot(h["step"], h["l0"], "o-", color=col, label=name)
    ax[2].plot(h["step"], h["dead"], "o-", color=col, label=name)
ax[0].set_title("fraction of variance explained"); ax[1].set_title("L0 (active latents / token)"); ax[1].set_yscale("log")
ax[2].set_title("dead latents (no fire in window)")
for a in ax: a.set_xlabel("step"); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ✔️ Checkpoint
assert hist["fve"][-1] > (0.6 if FAST else 0.8), "the TopK SAE should explain most of the variance"
assert abs(hist["l0"][-1] - TOPK_K) < 1e-6 or hist["l0"][-1] <= TOPK_K
print(f"✔️ TopK SAE: FVE {hist['fve'][-1]:.3f} at L0 = {hist['l0'][-1]:.0f}")

> **🍯 Key takeaway:** An SAE explains each activation as a sparse sum of dictionary directions. Judge it by **L0** (sparsity), **FVE** (fidelity) and **dead latents**; TopK sets L0 exactly, L1 only nudges it.

## Part 4 · What did it learn? Reading features 🔍

For every token in our corpus we compute the TopK code (sparse: 32 indices + values per token), then:

1. **Feature density**: how often each latent fires. Interpretable features tend to be rare (density $10^{-4}$–$10^{-2}$).
2. **Top-activating contexts**: the snippets where a feature fires most strongly. If they share an obvious theme, the feature is (at least partly) **monosemantic**.
3. **Logit effect**: push the feature's decoder direction through the unembedding, $W_U\,W_{\text{dec},i}$, to see which next tokens it promotes (ignoring the final layer norm — a rough but useful view).

In [ ]:
@torch.no_grad()
def encode_all(sae, X, batch=4096):
    vals, idxs = [], []
    for i in range(0, len(X), batch):
        pre = (X[i:i + batch] - sae.b_dec) @ sae.W_enc + sae.b_enc
        v, ix = pre.topk(sae.k, dim=-1)
        vals.append(F.relu(v).cpu()); idxs.append(ix.cpu())
    return torch.cat(vals), torch.cat(idxs)

code_vals, code_idx = encode_all(sae, X)
m_lat = sae.W_dec.shape[0]
fires = torch.zeros(m_lat).index_add_(0, code_idx.flatten(), (code_vals.flatten() > 0).float())
density = fires / len(X)
alive = density > 0
print(f"alive latents: {alive.float().mean():.1%} of {m_lat}")
plt.figure(figsize=(7, 3))
plt.hist(np.log10(density[alive].numpy()), bins=50, color=HIVE[0])
plt.xlabel("log10(firing frequency)"); plt.ylabel("# latents"); plt.title("SAE feature density"); plt.show()

def feature_acts(f):
    """Dense activation vector (n_tokens,) for latent f."""
    mask = code_idx == f
    return (code_vals * mask).sum(-1)

### 🧪 Your Turn #4 — top-activating contexts
Write `top_contexts(f, n=6, window=10)` returning a list of `(activation, context_string)` for the `n` strongest activations of latent `f`. Build the context from the `window` tokens before the activating token, and mark the activating token like `[[ token ]]`. Use `act_seq`, `act_pos` and `seqs` to locate each token in its sequence.

In [ ]:
def top_contexts(f, n=6, window=10):
    # TODO: a = feature_acts(f); top = a.topk(n)
    # TODO: for each index i: s, p = act_seq[i], act_pos[i]; left = seqs[s, max(0, p-window):p]; tok_i = seqs[s, p]
    # TODO: return [(activation, decoded_left + "[[" + decoded_tok + "]]"), ...]
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def top_contexts(f, n=6, window=10):
    a = feature_acts(f)
    vals, idx = a.topk(n)
    out = []
    for v, i in zip(vals.tolist(), idx.tolist()):
        s, p = act_seq[i].item(), act_pos[i].item()
        left = tok.decode(seqs[s, max(0, p - window):p])
        out.append((v, (left + "[[" + tok.decode(seqs[s, p:p + 1]) + "]]").replace("\n", "⏎")))
    return out

In [ ]:
# ✔️ Checkpoint
f0 = int(density.argmax())
ctx = top_contexts(f0, n=3)
assert len(ctx) == 3 and all("[[" in c and "]]" in c for _, c in ctx) and ctx[0][0] >= ctx[-1][0]
print("✔️ top_contexts OK\n")

@torch.no_grad()
def promoted_tokens(f, n=8):
    logit_dir = model.lm_head.weight @ sae.W_dec[f]          # (vocab,)
    return [tok.decode([t]) for t in logit_dir.topk(n).indices.tolist()]

# Show a handful of moderately rare features (dense ones are often boring, ultra-rare ones are noisy)
cands = torch.where((density > 2e-4) & (density < 2e-2))[0]
g = torch.Generator().manual_seed(0)
show = cands[torch.randperm(len(cands), generator=g)[:8]].tolist()
for f in show:
    print(f"━━ feature {f}  (density {density[f]:.4f})  promotes: {promoted_tokens(f, 6)}")
    for v, c in top_contexts(f, n=4):
        print(f"   {v:5.2f} | …{c[-90:]}")

Take a minute to label the features above. You'll usually find a mix of **token-level** features (fires on one word), **context** features (a topic: sports, military history, music…), and **grammatical** features (e.g. after a preposition, inside a number). Some will be uninterpretable — at this tiny scale (60k tokens, a few thousand latents) that's expected; production SAEs train on billions of tokens.

> **🍯 Key takeaway:** Features earn their names through evidence: consistent top-activating contexts and coherent promoted tokens. Many small-SAE features are interpretable, but not all.

## Part 5 · Does the SAE capture what the model *uses*? Splicing it in 🔌

FVE measures reconstruction in activation space, but some errors matter more than others. The stronger test: **replace** the residual stream at layer `LAYER` with the SAE's reconstruction during a forward pass, and measure the next-token cross-entropy on held-out text.

$$\text{loss recovered} = \frac{\mathcal{L}_\text{mean-ablate} - \mathcal{L}_\text{SAE}}{\mathcal{L}_\text{mean-ablate} - \mathcal{L}_\text{clean}}$$

(mean-ablation = replace the stream with its average: the model loses everything that layer carried.)

In [ ]:
held_ids = tok(text[N_TOKENS * 8: N_TOKENS * 8 + 40_000])["input_ids"]
n_held = min(16 if FAST else 48, len(held_ids) // SEQ_LEN)
if n_held < 4:                                # tiny fallback corpus: reuse the tail of the training sequences
    held = seqs[-8:]
else:
    held = torch.tensor(held_ids[: n_held * SEQ_LEN]).view(n_held, SEQ_LEN)

def make_replace_hook(fn):
    def hook(module, inputs, output):
        h = resid_hook_output(output)
        new = h.clone()
        new[:, 1:] = fn(h[:, 1:])            # leave the attention-sink position alone
        return (new,) + tuple(output[1:]) if isinstance(output, tuple) else new
    return hook

@torch.no_grad()
def ce_loss(replace_fn=None):
    handle = model.transformer.h[LAYER].register_forward_hook(make_replace_hook(replace_fn)) if replace_fn else None
    try:
        losses = [model(held[i:i + 8].to(DEVICE), labels=held[i:i + 8].to(DEVICE)).loss.item() for i in range(0, len(held), 8)]
    finally:
        if handle: handle.remove()
    return float(np.mean(losses))

def sae_reconstruct(s):
    return lambda h: s(((h - ACT_MEAN.to(h)) / ACT_SCALE))[0] * ACT_SCALE + ACT_MEAN.to(h)

clean = ce_loss()
mean_abl = ce_loss(lambda h: ACT_MEAN.to(h).expand_as(h))
topk_ce = ce_loss(sae_reconstruct(sae))
relu_ce = ce_loss(sae_reconstruct(relu_sae))
rec = lambda l: (mean_abl - l) / (mean_abl - clean)
print(f"clean CE {clean:.3f} | mean-ablated {mean_abl:.3f} | TopK-SAE spliced {topk_ce:.3f} ({rec(topk_ce):.1%} recovered) "
      f"| ReLU-SAE spliced {relu_ce:.3f} ({rec(relu_ce):.1%} recovered)")
plt.figure(figsize=(6, 3.2))
plt.bar(["clean", "TopK SAE", "ReLU SAE", "mean-ablate"], [clean, topk_ce, relu_ce, mean_abl], color=[HIVE[3], HIVE[0], HIVE[4], HIVE[2]])
plt.ylabel("next-token CE"); plt.title(f"Splicing the SAE into layer {LAYER}"); plt.show()
assert clean < topk_ce < mean_abl

> **🍯 Key takeaway:** The real test of an SAE is downstream: splice it into the model and see how much of the next-token loss it recovers.

## Part 6 · Steering: editing the residual stream during generation 🎛️

If concepts are directions, we can **add** a direction to push the model toward a concept:

$$h_\ell \leftarrow h_\ell + c\, v \quad \text{(at every position, every generation step)}.$$

Two ways to get $v$:
1. **Contrastive mean difference** (ActAdd, Turner et al., 2023; CAA, Rimsky et al., 2024): $v = \overline{h}_\ell(\text{positive prompts}) - \overline{h}_\ell(\text{negative prompts})$.
2. **An SAE feature's decoder direction** $W_{\text{dec},i}$ (the idea behind the "Golden Gate Claude" demo that accompanied *Scaling Monosemanticity*).

### 🧪 Your Turn #5 — a steering hook
Return a forward hook that adds `coef * vec` to the block's output hidden states (handle both tuple and tensor outputs, like `make_replace_hook` above).

In [ ]:
def make_steering_hook(vec, coef):
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def make_steering_hook(vec, coef):
    def hook(module, inputs, output):
        h = resid_hook_output(output)
        h = h + coef * vec.to(h)
        return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
    return hook

In [ ]:
# ✔️ Checkpoint
probe = tok("The weather today is", return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad():
    base_logits = model(probe).logits
    for coef, should_match in [(0.0, True), (5.0, False)]:
        h = model.transformer.h[LAYER].register_forward_hook(make_steering_hook(torch.randn(D_MODEL), coef))
        try:
            lg = model(probe).logits
        finally:
            h.remove()
        assert torch.allclose(lg, base_logits, atol=1e-5) == should_match
assert torch.allclose(model(probe).logits, base_logits), "hook must be removed afterwards"
print("✔️ steering hook OK (coef 0 is a no-op, hooks clean up)")

In [ ]:
POS_PROMPTS = ["I love this. It is wonderful and amazing", "What a fantastic, beautiful, happy day", "This is the best thing ever, I adore it",
               "Absolutely delightful and brilliant", "I am so happy and grateful", "Great job, excellent work, well done"]
NEG_PROMPTS = ["I hate this. It is awful and terrible", "What a horrible, ugly, miserable day", "This is the worst thing ever, I despise it",
               "Absolutely disgusting and dreadful", "I am so sad and angry", "Bad job, pathetic work, total failure"]

# Sentiment is easiest to steer fairly early in the network (we found layer ~N/6 works well for GPT-2;
# try other layers in the Boss Level). The SAE feature below uses our SAE's layer instead.
STEER_LAYER = max(1, N_LAYERS // 6)

@torch.no_grad()
def mean_resid(prompts, layer=STEER_LAYER):
    vecs, norms = [], []
    for p in prompts:
        r = collect_resid(tok(p, return_tensors="pt").input_ids, layer)[0, 1:]   # drop the sink position
        vecs.append(r.mean(0)); norms.append(r.norm(dim=-1).mean())
    return torch.stack(vecs).mean(0), torch.stack(norms).mean()

(pos_mean, pos_norm), (neg_mean, neg_norm) = mean_resid(POS_PROMPTS), mean_resid(NEG_PROMPTS)
diff = pos_mean - neg_mean
RESID_NORM = (pos_norm + neg_norm) / 2
steer_vec = diff / diff.norm() * RESID_NORM      # unit: "one typical residual-vector length"
print(f"layer {STEER_LAYER}: raw mean-difference norm {diff.norm():.1f} vs typical residual norm {RESID_NORM:.1f}")

POS_WORDS = set("love loved lovely wonderful amazing great happy best beautiful fantastic excellent good enjoy enjoyed fun delightful perfect brilliant glad nice awesome".split())
NEG_WORDS = set("hate hated awful terrible worst horrible bad sad angry ugly disgusting boring poor dreadful miserable pathetic failure annoying stupid".split())

def sentiment_score(texts):
    s = []
    for t in texts:
        w = re.findall(r"[a-z']+", t.lower())
        s.append((sum(x in POS_WORDS for x in w) - sum(x in NEG_WORDS for x in w)) / max(len(w), 1) * 100)
    return float(np.mean(s))

@torch.no_grad()
def generate_steered(prompts, vec=None, coef=0.0, n=4, max_new_tokens=30, seed=0, layer=LAYER):
    handle = model.transformer.h[layer].register_forward_hook(make_steering_hook(vec, coef)) if vec is not None and coef else None
    outs = []
    try:
        for p in prompts:
            torch.manual_seed(seed)
            enc = tok([p] * n, return_tensors="pt").to(DEVICE)
            g = model.generate(**enc, do_sample=True, top_p=0.9, temperature=0.8, max_new_tokens=max_new_tokens,
                               pad_token_id=tok.eos_token_id)
            outs += [tok.decode(x[enc.input_ids.shape[1]:], skip_special_tokens=True) for x in g]
    finally:
        if handle: handle.remove()
    return outs

### 🔮 Predict-then-run
We'll generate from neutral prompts ("I went to see the new movie and", …) while adding $c \cdot v$ at layer `STEER_LAYER`, where $v$ is the mean difference rescaled to the length of a typical residual vector and $c \in \{-0.6, \dots, 0.6\}$. What happens to the sentiment score as $c$ grows? What happens to *fluency* at the extremes?

In [ ]:
NEUTRAL = ["I went to see the new movie and", "My neighbour's dog is", "The food at the restaurant was"]
COEFS = [-0.6, -0.3, 0, 0.3, 0.6] if not FAST else [-0.6, 0, 0.6]
scores, samples = [], {}
for c in COEFS:
    gens = generate_steered(NEUTRAL, steer_vec, c, n=3 if FAST else 6, layer=STEER_LAYER)
    scores.append(sentiment_score(gens)); samples[c] = gens
    print(f"coef {c:+.1f} | sentiment {scores[-1]:+.2f} | e.g. {NEUTRAL[0]!r} →{gens[0][:100]!r}")
plt.figure(figsize=(6, 3.2)); plt.plot(COEFS, scores, "o-", color=HIVE[3]); plt.axhline(0, color=HIVE[1], lw=0.8)
plt.xlabel("steering coefficient c"); plt.ylabel("(pos − neg words) per 100 words"); plt.title("Mean-difference sentiment steering")
plt.show()

**Reading the curve.** Negative coefficients pull the sentiment score down sharply; positive ones push it up, but less dramatically, because GPT-2's continuations of these prompts are already mildly positive. Push much past $|c| \approx 0.6$ and fluency collapses into repetition. Try the same experiment at the SAE layer (`layer=LAYER`): steering there is noticeably weaker for sentiment, a reminder that *where* you intervene matters. Our word-list metric is crude — a real study would use a sentiment classifier and also measure perplexity to quantify the fluency cost.

In [ ]:
# ✔️ Checkpoint — positive steering should be more positive than negative steering
assert scores[-1] > scores[0], scores
print("✔️ steering moves sentiment in the intended direction")

### Steering with an SAE feature
Now pick a feature and add its decoder direction. We look for a latent that is **selective for a topic**: from a few candidate topics (sports, music, war, ships, weather) we take the one most represented in our activation corpus, then choose the latent whose average activation on that topic's words is highest relative to everywhere else. The decoder row lives in *normalized* activation space, so we multiply by `ACT_SCALE` to put it in raw residual units (coefficient 1 ≈ the size of a typical residual vector).

In [ ]:
TOPICS = {
    "sports":  [" season", " team", " goals", " football", " match", " league", " players", " coach", " championship"],
    "music":   [" song", " album", " music", " band", " singer", " guitar", " concert", " songs", " chart"],
    "war":     [" battle", " army", " troops", " war", " soldiers", " regiment", " forces", " attack", " enemy"],
    "ships":   [" ship", " navy", " fleet", " naval", " ships", " guns", " sailed", " crew", " port"],
    "weather": [" storm", " hurricane", " winds", " rain", " tropical", " flooding", " weather", " landfall", " cyclone"],
}
def topic_token_ids(words):
    return [tok(w).input_ids[0] for w in words if len(tok(w).input_ids) == 1]
counts = {t: int(torch.isin(act_tokens, torch.tensor(topic_token_ids(ws))).sum()) for t, ws in TOPICS.items()}
TOPIC = max(counts, key=counts.get)
print("topic word counts in corpus:", counts, "→ using", repr(TOPIC))
is_topic = torch.isin(act_tokens, torch.tensor(topic_token_ids(TOPICS[TOPIC])))

# selectivity = mean activation on topic tokens / (mean activation elsewhere + eps), among reasonably common latents
on = torch.zeros(m_lat).index_add_(0, code_idx[is_topic].flatten(), code_vals[is_topic].flatten()) / max(int(is_topic.sum()), 1)
off = torch.zeros(m_lat).index_add_(0, code_idx[~is_topic].flatten(), code_vals[~is_topic].flatten()) / max(int((~is_topic).sum()), 1)
selectivity = torch.where(density > 1e-4, on / (off + 1e-3), torch.zeros_like(on))
feat = int(selectivity.argmax())
print(f"chosen feature {feat}: selectivity {selectivity[feat]:.1f}, density {density[feat]:.4f}, promotes {promoted_tokens(feat, 8)}")
for v, c in top_contexts(feat, n=5):
    print(f"   {v:5.2f} | …{c[-90:]}")

feat_dir = sae.W_dec[feat].detach().cpu() * ACT_SCALE
TOPIC_SET = {w.strip() for w in TOPICS[TOPIC]}
def topic_rate(texts):
    w = [x for t in texts for x in re.findall(r"[a-z]+", t.lower())]
    return 100 * sum(x in TOPIC_SET for x in w) / max(len(w), 1)

FEAT_COEFS = [0, 1, 2] if FAST else [0, 0.5, 1, 1.5, 2]
rates = []
for c in FEAT_COEFS:
    gens = generate_steered(NEUTRAL, feat_dir, c, n=2 if FAST else 4)
    rates.append(topic_rate(gens))
    print(f"coef {c} | {TOPIC} words per 100: {rates[-1]:.2f} | e.g. →{gens[0][:110]!r}")
plt.figure(figsize=(6, 3.2)); plt.plot(FEAT_COEFS, rates, "o-", color=HIVE[0])
plt.xlabel("coefficient (× ACT_SCALE × decoder direction)"); plt.ylabel(f"{TOPIC} words per 100"); plt.title(f"Steering with SAE feature {feat}")
plt.show()

**Interpretation & caveats.**
- If the feature is genuinely about the topic, generations drift toward its vocabulary as the coefficient grows — and eventually degrade into repetition. Too large a push knocks the residual stream off the manifold the later layers expect.
- Mean-difference vectors are cheap and often work surprisingly well; SAE features give you a **named, inspectable** direction, but a small SAE's features can be messy or split ("feature splitting" across dictionary sizes is a known phenomenon).
- Steering is a research tool for testing causal hypotheses ("does this direction *cause* this behavior?"), and increasingly a monitoring/control tool — but evaluate side effects before trusting it.

> **🍯 Key takeaway:** Adding a direction to the residual stream changes behaviour, which is causal evidence that the direction means what you think. It is also a control knob with real side effects on fluency.

## 🏆 Boss Level
1. **The sparsity–fidelity frontier.** Train TopK SAEs with $k \in \{8, 16, 32, 64\}$ and ReLU SAEs with several $\lambda$. Plot FVE (and spliced CE loss) against L0. Which architecture dominates at equal L0? (Gao et al. 2024 report TopK does.)
2. **Feature splitting.** Train SAEs with expansion 2× and 16×. Find a feature in the small SAE and its nearest decoder directions (cosine) in the big one. Do they split one concept into finer sub-concepts?
3. **Automated interpretability.** With an LLM API key, send each feature's top contexts to a model and ask for a one-line label, then *score* the label by asking the model to predict activations on held-out contexts (the "explain then simulate" protocol of Bills et al., 2023).

## 🍯 Key Takeaways
- **Residual stream = shared bus.** Blocks add their outputs to it; the logit lens reads it early.
- **Hooks are the interpretability toolkit.** Read with `return None`, overwrite with `return new_output`, and always remove them.
- **Superposition → SAEs.** Models pack more features than dimensions, and a wide, sparse dictionary pulls them apart.
- **Measure three things:** L0 (sparsity), FVE (reconstruction) and dead latents, plus **loss recovered** when the SAE is spliced back in.
- **TopK vs L1:** TopK fixes L0 exactly and avoids shrinkage; L1 is the classic, tunable baseline.
- **Interpret with evidence:** top-activating contexts *and* logit effects, then test causally with ablation or steering.
- **Steering works and has costs:** mean-difference vectors are cheap, SAE features are named, and both can wreck fluency if pushed too hard.

## 🎤 Explain it to a friend
Explain to a friend why looking at one "neuron" inside a language model is like listening to one microphone in a room where twenty people talk at once, and how a sparse autoencoder acts like a set of noise-cancelling headphones that separate the voices. Then explain how you checked that a separated "voice" really means something: you turned its volume up and watched what the model started talking about. One paragraph, no equations.

## 💼 Interview corner
**Q1. What is superposition, and why does it motivate sparse autoencoders?**
<details><summary>Answer</summary>

Superposition is the hypothesis that a network represents more features than it has neurons or dimensions, by assigning features to nearly-orthogonal directions and relying on their sparsity (few are active at once) to keep interference low. As a result individual neurons are polysemantic. An SAE learns an overcomplete dictionary ($m \gg d$) with a sparsity constraint, so each activation is explained by a few dictionary directions. Those directions are candidates for the underlying monosemantic features.
</details>

**Q2. Compare ReLU+L1 SAEs with TopK SAEs. What are dead latents and why do they matter?**
<details><summary>Answer</summary>

ReLU+L1 adds $\lambda\lVert z\rVert_1$ to the reconstruction loss. Sparsity is controlled only indirectly through $\lambda$, and the L1 penalty shrinks the magnitude of active latents (biasing reconstructions). TopK keeps exactly the $k$ largest pre-activations, which sets L0 directly, removes shrinkage, and in Gao et al. (2024) gave a better sparsity–fidelity frontier. Dead latents never activate on any input: wasted capacity that grows with dictionary size. Mitigations include careful init, resampling dead latents, and auxiliary losses (e.g. AuxK).
</details>

**Q3. You found an SAE feature that seems to mean "sports". How would you check it is causally meaningful?**
<details><summary>Answer</summary>

Correlational checks first: top-activating contexts across many examples, activation on held-out sports vs non-sports text, and the tokens its decoder direction promotes through the unembedding. Then causal checks: **ablate** it (set the latent to zero in the spliced SAE) and see whether sports-related predictions get worse while others don't; **steer** with its decoder direction and see whether generations shift toward sports. Measure side effects (perplexity, other topics) and compare against random directions of the same norm as a control.
</details>

## 🐝 Fun fact
In May 2024, alongside *Scaling Monosemanticity*, Anthropic briefly released **"Golden Gate Claude"**: Claude 3 Sonnet with a single SAE feature for the Golden Gate Bridge clamped to a high value. The model brought the bridge into almost every answer, and when asked about its physical form it claimed to *be* the Golden Gate Bridge. That one feature, among millions, changed its whole self-description.

## 📚 Further reading
- Bricken et al. (2023). *Towards Monosemanticity: Decomposing Language Models With Dictionary Learning.* Transformer Circuits Thread.
- Templeton et al. (2024). *Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet.* Transformer Circuits Thread.
- Gao et al. (2024). *Scaling and evaluating sparse autoencoders.* (TopK SAEs)
- Cunningham et al. (2023). *Sparse Autoencoders Find Highly Interpretable Features in Language Models.*
- Elhage et al. (2022). *Toy Models of Superposition.*
- Lieberum et al. (2024). *Gemma Scope: Open Sparse Autoencoders Everywhere All At Once on Gemma 2.*
- nostalgebraist (2020). *interpreting GPT: the logit lens* (LessWrong).
- Belrose et al. (2023). *Eliciting Latent Predictions from Transformers with the Tuned Lens.*
- Turner et al. (2023). *Steering Language Models With Activation Engineering.* (ActAdd)
- Rimsky et al. (2024). *Steering Llama 2 via Contrastive Activation Addition.*
- Bills et al. (2023). *Language models can explain neurons in language models.* (OpenAI)
- Tools: SAELens, TransformerLens, Neuronpedia.